<a href="https://colab.research.google.com/github/81Miralay81/Duzce-Anma-Video-Studio/blob/main/colab/KANATIX_Colab_Free_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KANATIX — Ücretsiz Google Colab Wan 2.2 üretim motoru

Bu not defteri KANATIX Android uygulamasına geçici, ücretsiz bir üretim API'si sağlar. Kiralık sunucu, ücretli API veya Colab Pro gerektirmez.

## Kullanım
1. Bu dosyayı Google Colab'da açın.
2. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin (varsa).
3. Hücreleri sırayla çalıştırın. İlk kurulum model dosyalarını indirir; uzun sürebilir.
4. Google Drive erişim iznini onaylayın.
5. Son hücrede görünen **KANATIX API adresini** ve **erişim anahtarını** Android uygulamasındaki Colab bağlantı ayarlarına girin.
6. Colab sekmesini açık tutun. Oturum kapanırsa yeni adres ve anahtar üretileceği için APK ayarlarını güncelleyin.

## Model ve sınırlar
- Wan 2.2 I2V 14B, Q4_K_M GGUF + LightX2V LoRA; 480p dikey, yaklaşık 4 saniyelik sahneler için ayarlanmıştır.
- Topluluk notebook'u T4 üzerinde 480p kısa klip üretimi bildirmektedir; gerçek hız ve çalışabilirlik Google'ın verdiği GPU'ya, kotaya ve oturum belleğine bağlıdır.
- Ücretsiz Colab GPU garantili değildir. T4 atanmazsa veya VRAM yetersizse üretim başlamayabilir.
- Tamamlanan MP4 dosyaları Google Drive / MyDrive/Kanatix/ColabJobs klasörüne de kopyalanır.
- Colab ücretsiz oturumu kalıcı sunucu değildir. Kapanınca API bağlantısı kesilir; dosyalar Drive'da kalır.

## Kaynak
Kurulum ve model pipeline'ı, [Collab-AI-Gen Wan 2.2 Lightx2v notebook](https://github.com/Collab-AI-Gen/Google-Colab_Notebooks/blob/main/wan2_2/wan22_Lightx2v_Aigen.ipynb) projesinin ilk kurulum hücresini çalışma anında alır. Üretim katmanı KANATIX için Flask + Cloudflare Quick Tunnel API'sidir. Kaynak notebook'u çalıştırmadan önce inceleyebilirsiniz.


In [ ]:
# GPU doğrula ve upstream Wan 2.2 Lightx2v kurulumunu hazırla.
# Not: Model indirmeleri yaklaşık onlarca GB olabilir.
import requests, json, re, os, textwrap
from IPython.display import display, Markdown

UPSTREAM = "https://raw.githubusercontent.com/Collab-AI-Gen/Google-Colab_Notebooks/main/wan2_2/wan22_Lightx2v_Aigen.ipynb"
r = requests.get(UPSTREAM, timeout=60)
r.raise_for_status()
upstream_nb = r.json()
setup_cell = next(c for c in upstream_nb["cells"] if c.get("cell_type") == "code")
setup_source = "".join(setup_cell["source"])

# Gereksiz video-frame interpolation kurulumu çıkarılır; KANATIX kendi MP4 doğrulamasını yapar.
start = setup_source.find("%cd /content\n!git clone https://github.com/Isi-dev/Practical-RIFE")
end = setup_source.find("%cd /content/ComfyUI\n!apt -y install -qq aria2 ffmpeg", start)
if start >= 0 and end > start:
    setup_source = setup_source[:start] + setup_source[end:]

# Upstream hücresindeki pip'e ait olmayan -y seçeneğini düzelt.
setup_source = setup_source.replace("!pip install onnxruntime-gpu -y", "!pip install onnxruntime-gpu")

# Opsiyonel hareket LoRA'ları indirilmez; KANATIX nötr prompt_assist kullanır.
setup_lines = []
for line in setup_source.splitlines():
    if any(line.lstrip().startswith(name + " = model_download(") for name in ("walkingToViewersL", "walkingFromBehindL", "dancingL")):
        continue
    setup_lines.append(line)
setup_source = "\n".join(setup_lines) + "\n"
setup_source += "\nwalkingToViewersL = walkingFromBehindL = dancingL = None\n"
print("GPU bilgisi:")
!nvidia-smi
print("Wan 2.2 kurulum hücresi çalıştırılıyor. Model indirmeleri zaman alabilir.")
get_ipython().run_cell(setup_source)


In [ ]:
# KANATIX API — ücretsiz Colab oturumunu APK'ya aç
!pip -q install flask
from google.colab import drive
drive.mount('/content/drive')

import os, re, time, uuid, shutil, secrets, threading, subprocess, glob, hmac, json
from flask import Flask, request, jsonify, send_file

KANATIX_TOKEN = secrets.token_urlsafe(32)
KANATIX_DRIVE = "/content/drive/MyDrive/Kanatix/ColabJobs"
os.makedirs(KANATIX_DRIVE, exist_ok=True)
os.makedirs("/content/kanatix_uploads", exist_ok=True)
generation_lock = threading.Lock()
api = Flask("kanatix_colab")
api.config["MAX_CONTENT_LENGTH"] = 25 * 1024 * 1024

@api.get("/health")
def health():
    import torch
    return jsonify({"ok": True, "service": "KANATIX Wan 2.2 Colab", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU yok"})

@api.post("/generate")
def generate():
    supplied = request.headers.get("X-Kanatix-Token", "")
    if not hmac.compare_digest(supplied, KANATIX_TOKEN):
        return jsonify({"error": "Yetkisiz erişim"}), 401
    if not generation_lock.acquire(blocking=False):
        return jsonify({"error": "GPU şu anda başka bir sahne üretiyor. Biraz bekleyin."}), 429
    try:
        image = request.files.get("image")
        if image is None:
            return jsonify({"error": "image alanı eksik"}), 400
        if image.mimetype not in ("image/jpeg", "image/png", "image/webp"):
            return jsonify({"error": "Yalnızca JPG, PNG veya WEBP kabul edilir."}), 415
        prompt = (request.form.get("prompt") or "").strip()
        if not prompt:
            return jsonify({"error": "prompt alanı boş"}), 400
        seed = int(request.form.get("seed", "0") or "0")
        if seed <= 0:
            seed = int(time.time()) % 2147483647
        job = uuid.uuid4().hex
        ext = ".png" if image.mimetype == "image/png" else (".webp" if image.mimetype == "image/webp" else ".jpg")
        input_path = f"/content/kanatix_uploads/{job}{ext}"
        image.save(input_path)
        print(f"KANATIX job başladı: {job}")
        # Kaynak notebook'un generate_video fonksiyonu Wan2.2 I2V Q4_K_M + LightX2V kullanır.
        # Her istek öncesi eski global çıktıyı sıfırla; önceki sahnenin dosyasını yanlışlıkla gönderme.
        global output_path
        output_path = None
        generation_started = time.time()
        generate_video(
            image_path=input_path,
            positive_prompt=prompt,
            prompt_assist="none",
            negative_prompt="static image, frozen frame, blurry, deformed anatomy, extra limbs, duplicate people, modern objects, fantasy, text, watermark",
            width=480, height=832,
            seed=seed,
            steps=6, cfg_scale=1.0,
            sampler_name="euler", scheduler="simple",
            frames=65, fps=16, output_format="mp4",
            overwrite=False,
            use_lora=False, use_lora2=False, use_lora3=False,
            use_lightx2v=True, lightx2v_Strength=3.0, lightx2v_steps=6,
            use_pusa=False, use_sage_attention=True,
            enable_flow_shift=True, shift=8.0,
            enable_flow_shift2=True, shift2=8.0,
            end_step1=3
        )
        video_path = globals().get("output_path")
        # Bazı notebook sürümlerinde global çıktı yolu güncellenmezse bu işin ürettiği en yeni MP4'ü bul.
        if not video_path or not os.path.isfile(video_path):
            output_dir = "/content/ComfyUI/output"
            candidates = [
                p for p in glob.glob(os.path.join(output_dir, "*.mp4"))
                if os.path.getmtime(p) >= generation_started - 2
            ]
            video_path = max(candidates, key=os.path.getmtime) if candidates else None
        if not video_path or not os.path.isfile(video_path) or os.path.getsize(video_path) < 2048:
            raise RuntimeError(f"Wan üretimi tamamlandı ancak bu işe ait MP4 bulunamadı. output_path={video_path!r}")
        with open(video_path, "rb") as check_file:
            source_header = check_file.read(16)
        if len(source_header) < 12 or source_header[4:8] != b"ftyp":
            raise RuntimeError(
                f"Wan çıktı dosyası MP4 değil. Dosya={video_path}; "
                f"başlık={source_header[:16]!r}. Eski veya yanlış çıktı dosyası gönderilmeyecek."
            )
        # MP4'ü Android uyumlu H.264/yuv420p biçiminde normalize et.
        normalized_dir = "/content/kanatix_outputs"
        os.makedirs(normalized_dir, exist_ok=True)
        normalized_path = os.path.join(normalized_dir, f"{job}.mp4")
        ffmpeg = shutil.which("ffmpeg") or "/usr/bin/ffmpeg"
        transcode = subprocess.run(
            [ffmpeg, "-y", "-i", video_path, "-map", "0:v:0", "-an",
             "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
             "-pix_fmt", "yuv420p", "-movflags", "+faststart", normalized_path],
            capture_output=True, text=True, timeout=300
        )
        if transcode.returncode != 0 or not os.path.isfile(normalized_path) or os.path.getsize(normalized_path) < 2048:
            raise RuntimeError("MP4 dönüştürme başarısız: " + (transcode.stderr or "ffmpeg çıktı üretmedi")[-1200:])
        with open(normalized_path, "rb") as check_file:
            normalized_header = check_file.read(16)
        if normalized_header[4:8] != b"ftyp":
            raise RuntimeError(f"Normalize edilen dosya MP4 başlığı taşımıyor: {normalized_header[:16]!r}")
        saved_path = os.path.join(KANATIX_DRIVE, f"{job}.mp4")
        shutil.copy2(normalized_path, saved_path)
        print(f"KANATIX job tamamlandı: {job} -> {saved_path}")
        return send_file(normalized_path, mimetype="video/mp4", as_attachment=True,
                         download_name=f"kanatix-{job}.mp4", conditional=False)
    except Exception as exc:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(exc)[:1200], "type": type(exc).__name__}), 500
    finally:
        generation_lock.release()

def _run_api():
    api.run(host="0.0.0.0", port=7860, threaded=True, use_reloader=False)

threading.Thread(target=_run_api, daemon=True).start()
print("KANATIX API başlatıldı.")

# Hesap gerektirmeyen Cloudflare Quick Tunnel
cloudflared = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cloudflared], check=True)
    os.chmod(cloudflared, 0o755)
log_path = "/content/kanatix-cloudflared.log"
log_file = open(log_path, "w")
tunnel = subprocess.Popen([cloudflared, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"], stdout=log_file, stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(2)
    try:
        log_file.flush()
        log_text = open(log_path, "r").read()
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            public_url = match.group(0)
            break
    except Exception:
        pass
if not public_url:
    raise RuntimeError("Cloudflare ücretsiz tünel adresi alınamadı. Hücre çıktısını ve /content/kanatix-cloudflared.log dosyasını kontrol edin.")

print("\n" + "="*58)
# API adresi ve erişim anahtarını yalnızca kullanıcının kendi Google Drive alanına kaydet.
KANATIX_CONFIG_DIR = "/content/drive/MyDrive/Kanatix/Config"
os.makedirs(KANATIX_CONFIG_DIR, exist_ok=True)
KANATIX_CONFIG_JSON = os.path.join(KANATIX_CONFIG_DIR, "colab_api.json")
KANATIX_CONFIG_TXT = os.path.join(KANATIX_CONFIG_DIR, "colab_api.txt")
config_data = {
    "service": "KANATIX Wan 2.2 Colab",
    "api_url": public_url.rstrip("/"),
    "token": KANATIX_TOKEN,
    "health_url": public_url.rstrip("/") + "/health",
    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "gpu": __import__("torch").cuda.get_device_name(0) if __import__("torch").cuda.is_available() else "GPU yok"
}
with open(KANATIX_CONFIG_JSON, "w", encoding="utf-8") as config_file:
    json.dump(config_data, config_file, ensure_ascii=False, indent=2)
with open(KANATIX_CONFIG_TXT, "w", encoding="utf-8") as config_file:
    config_file.write("KANATIX API ADRESI: " + config_data["api_url"] + "\\n")
    config_file.write("ERISIM ANAHTARI: " + KANATIX_TOKEN + "\\n")
    config_file.write("SAGLIK KONTROLU: " + config_data["health_url"] + "\\n")
    config_file.write("GUNCELLEME: " + config_data["updated_at"] + "\\n")
print("API bilgileri Google Drive'a kaydedildi:")
print(KANATIX_CONFIG_JSON)
print(KANATIX_CONFIG_TXT)

print("KANATIX ÜCRETSİZ COLAB API HAZIR")
print("API ADRESİ:", public_url)
print("ERİŞİM ANAHTARI:", KANATIX_TOKEN)
print("Bu iki bilgiyi APK > Colab Bağlantısı bölümüne girin.")
print("Oturum kapanana kadar Colab sekmesini açık tutun.")
print("Drive çıktı klasörü:", KANATIX_DRIVE)
print("="*58)


## Drive kayıt kurtarma hücresi

API hücresi çalıştı ancak Drive'a kayıt aşamasında hata verdi ise, **API hücresini yeniden çalıştırmadan** aşağıdaki hücreyi tek başına çalıştırın. Bu hücre mevcut `public_url` ve `KANATIX_TOKEN` değişkenlerini kullanır; model indirmez, API veya tünel başlatmaz.

In [ ]:
# Yalnızca API ve Cloudflare tüneli aynı Colab oturumunda zaten çalışıyorsa kullanın.
import os, json, time

if not globals().get("public_url") or not globals().get("KANATIX_TOKEN"):
    raise RuntimeError("Mevcut API adresi veya anahtarı bulunamadı. API hücresinin çalıştığı aynı oturumda olduğunuzu kontrol edin; model kurulumunu tekrar başlatmayın.")

KANATIX_CONFIG_DIR = "/content/drive/MyDrive/Kanatix/Config"
os.makedirs(KANATIX_CONFIG_DIR, exist_ok=True)
api_url = public_url.rstrip("/")
config_data = {
    "service": "KANATIX Wan 2.2 Colab",
    "api_url": api_url,
    "token": KANATIX_TOKEN,
    "health_url": api_url + "/health",
    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "gpu": __import__("torch").cuda.get_device_name(0) if __import__("torch").cuda.is_available() else "GPU yok"
}

json_path = os.path.join(KANATIX_CONFIG_DIR, "colab_api.json")
txt_path = os.path.join(KANATIX_CONFIG_DIR, "colab_api.txt")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(config_data, f, ensure_ascii=False, indent=2)
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("KANATIX API ADRESI: " + api_url + "\n")
    f.write("ERISIM ANAHTARI: " + KANATIX_TOKEN + "\n")
    f.write("SAGLIK KONTROLU: " + config_data["health_url"] + "\n")
    f.write("GUNCELLEME: " + config_data["updated_at"] + "\n")

print("DRIVE KAYDI TAMAMLANDI")
print("API adresi:", api_url)
print("Dosyalar:", json_path, "|", txt_path)
print("Güvenlik: erişim anahtarını kimseyle paylaşmayın; bu hücre anahtarı ekrana yazdırmaz.")

In [ ]:
# KANATIX API SICAK DÜZELTME
# Bu hücre mevcut Flask sunucusunun /generate rotasını değiştirir.
# Modeli, Flask sunucusunu veya Cloudflare tünelini yeniden başlatmaz.
import os, glob, time, uuid, shutil, subprocess, hmac
from flask import request, jsonify, send_file

def kanatix_generate_fixed():
    supplied = request.headers.get("X-Kanatix-Token", "")
    if not hmac.compare_digest(supplied, KANATIX_TOKEN):
        return jsonify({"error": "Yetkisiz erişim"}), 401
    if not generation_lock.acquire(blocking=False):
        return jsonify({"error": "GPU şu anda başka bir sahne üretiyor. Biraz bekleyin."}), 429
    try:
        image = request.files.get("image")
        if image is None:
            return jsonify({"error": "image alanı eksik"}), 400
        if image.mimetype not in ("image/jpeg", "image/png", "image/webp"):
            return jsonify({"error": "Yalnızca JPG, PNG veya WEBP kabul edilir."}), 415
        prompt = (request.form.get("prompt") or "").strip()
        if not prompt:
            return jsonify({"error": "prompt alanı boş"}), 400
        seed = int(request.form.get("seed", "0") or "0")
        if seed <= 0:
            seed = int(time.time()) % 2147483647
        job = uuid.uuid4().hex
        ext = ".png" if image.mimetype == "image/png" else (".webp" if image.mimetype == "image/webp" else ".jpg")
        input_path = f"/content/kanatix_uploads/{job}{ext}"
        image.save(input_path)
        global output_path
        output_path = None
        generation_started = time.time()
        print(f"KANATIX job başladı: {job}")
        generate_video(
            image_path=input_path, positive_prompt=prompt, prompt_assist="none",
            negative_prompt="static image, frozen frame, blurry, deformed anatomy, extra limbs, duplicate people, modern objects, fantasy, text, watermark",
            width=480, height=832, seed=seed, steps=6, cfg_scale=1.0,
            sampler_name="euler", scheduler="simple", frames=65, fps=16,
            output_format="mp4", overwrite=False,
            use_lora=False, use_lora2=False, use_lora3=False,
            use_lightx2v=True, lightx2v_Strength=3.0, lightx2v_steps=6,
            use_pusa=False, use_sage_attention=True,
            enable_flow_shift=True, shift=8.0,
            enable_flow_shift2=True, shift2=8.0, end_step1=3
        )
        video_path = globals().get("output_path")
        if not video_path or not os.path.isfile(video_path):
            candidates = [p for p in glob.glob("/content/ComfyUI/output/*.mp4")
                          if os.path.getmtime(p) >= generation_started - 2]
            video_path = max(candidates, key=os.path.getmtime) if candidates else None
        if not video_path or not os.path.isfile(video_path) or os.path.getsize(video_path) < 2048:
            raise RuntimeError(f"Wan üretimi tamamlandı ancak bu işe ait MP4 bulunamadı. output_path={video_path!r}")
        with open(video_path, "rb") as f:
            header = f.read(16)
        if len(header) < 12 or header[4:8] != b"ftyp":
            raise RuntimeError(f"Wan çıktı dosyası MP4 değil. Dosya={video_path}; başlık={header[:16]!r}")
        normalized_dir = "/content/kanatix_outputs"
        os.makedirs(normalized_dir, exist_ok=True)
        normalized_path = os.path.join(normalized_dir, f"{job}.mp4")
        ffmpeg = shutil.which("ffmpeg") or "/usr/bin/ffmpeg"
        result = subprocess.run(
            [ffmpeg, "-y", "-i", video_path, "-map", "0:v:0", "-an",
             "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
             "-pix_fmt", "yuv420p", "-movflags", "+faststart", normalized_path],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode != 0 or not os.path.isfile(normalized_path) or os.path.getsize(normalized_path) < 2048:
            raise RuntimeError("MP4 dönüştürme başarısız: " + (result.stderr or "ffmpeg çıktı üretmedi")[-1200:])
        with open(normalized_path, "rb") as f:
            if f.read(12)[4:8] != b"ftyp":
                raise RuntimeError("Normalize edilen dosyanın MP4 başlığı hatalı.")
        saved_path = os.path.join(KANATIX_DRIVE, f"{job}.mp4")
        shutil.copy2(normalized_path, saved_path)
        print(f"KANATIX job tamamlandı: {job} -> {saved_path}")
        return send_file(normalized_path, mimetype="video/mp4", as_attachment=True,
                         download_name=f"kanatix-{job}.mp4", conditional=False)
    except Exception as exc:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(exc)[:1200], "type": type(exc).__name__}), 500
    finally:
        generation_lock.release()

api.view_functions["generate"] = kanatix_generate_fixed
print("KANATIX API SICAK DÜZELTME UYGULANDI. /generate rotası MP4 başlığını doğrulayıp H.264'e dönüştürecek.")
